In [32]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args, **kwargs):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80), **kwargs)
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

result=load_dotenv('../openApiKey.env')  # reads .env file in the current directory

#print("Loaded:",result)
api_key = os.getenv("OPENAI_API_KEY")
print(os.getcwd())

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

/Users/anupagarwal/Documents/PersonalFolder/ScalerAICourse/Practicals/workingwithllms/lec_25_why_mcp_is_needed
API key loaded successfully.


In [ ]:
# Install the bits we need
#!pip install -q mcp openai

In [ ]:
#pip install -q mcp openai


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


async def main():
    async with streamable_http_client(
        "http://127.0.0.1:8001/mcp"
    ) as (read, write, _):

        async with ClientSession(read, write) as session:
            await session.initialize()

            # List tools
            tools = await session.list_tools()

            print("Tools exposed by the math server:")
            for tool in tools.tools:
                print(f"  • {tool.name} — {tool.description}")
                result = await session.call_tool(
                tool.name,
                {"a": 6, "b": 7}
                )
                print(f"       {tool.name}(6, 7) ={result.content[0].text}" )

            # Call multiply
            # result = await session.call_tool(
            #     "multiply",
            #     {"a": 6, "b": 7}
            # )

#            print("\nmultiply(6, 7) =", result.content[0].text)


await main()

Tools exposed by the math server:
  • add — Add two numbers.
       add(6, 7) =13.0
  • multiply — Multiply two numbers.
       multiply(6, 7) =42.0
  • power — Raise base to the given exponent.
       power(6, 7) =279936.0
  • modulo — Return the remainder of a divided by b.
       modulo(6, 7) =6.0


In [105]:
import json
from contextlib import AsyncExitStack
from mcp.client.streamable_http import streamable_http_client


class MCPHost_http:
    """Manages multiple MCP client sessions and exposes their tools as one pool."""

    def __init__(self):
        self.session_by_tool = {}   # tool_name -> ClientSession
        self.openai_tools = []      # tool schemas in OpenAI Responses format
        self._stack = AsyncExitStack()

    async def add_server(self, url):
        """Spawn a server and register all its tools."""
        read, write, _ = await self._stack.enter_async_context(streamable_http_client(url))
        session = await self._stack.enter_async_context(ClientSession(read, write))
        
        await session.initialize()
        # List tools
        listed  = await session.list_tools()
        for t in listed.tools:
            self.session_by_tool[t.name] = session
            # Translate MCP's tool shape -> OpenAI Responses tool shape.
            # MCP gives us: name, description, inputSchema (JSON schema).
            # OpenAI wants: type, name, description, parameters.
            self.openai_tools.append({
                "type": "function",
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            })
        print(f"  connected — tools: {[t.name for t in listed.tools]}")
        
        
    async def call(self, tool_name, args):
        """Route a tool call to the right server."""
        print(f"tool name which is asked for calling is {tool_name}")
        session = self.session_by_tool[tool_name]
        
        result = await session.call_tool(tool_name, args)
        # result.content is a list of content parts; grab their text
        return "\n".join(c.text for c in result.content if hasattr(c, "text"))

    async def close(self):
        await self._stack.aclose()


In [91]:
import json
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


class MCPHost:
    """Manages multiple MCP client sessions and exposes their tools as one pool."""

    def __init__(self):
        self.session_by_tool = {}   # tool_name -> ClientSession
        self.openai_tools = []      # tool schemas in OpenAI Responses format
        self._stack = AsyncExitStack()

    async def add_server(self, command, args):
        """Spawn a server and register all its tools."""
        params = StdioServerParameters(command=command, args=args)
        read, write = await self._stack.enter_async_context(stdio_client(params))
        session = await self._stack.enter_async_context(ClientSession(read, write))
        await session.initialize()

        listed = await session.list_tools()
        for t in listed.tools:
            self.session_by_tool[t.name] = session
            # Translate MCP's tool shape -> OpenAI Responses tool shape.
            # MCP gives us: name, description, inputSchema (JSON schema).
            # OpenAI wants: type, name, description, parameters.
            self.openai_tools.append({
                "type": "function",
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            })
        print(f"  connected — tools: {[t.name for t in listed.tools]}")

    async def call(self, tool_name, args):
        """Route a tool call to the right server."""
        print(f"tool name which is asked for calling is {tool_name}")
        session = self.session_by_tool[tool_name]
        
        result = await session.call_tool(tool_name, args)
        # result.content is a list of content parts; grab their text
        return "\n".join(c.text for c in result.content if hasattr(c, "text"))

    async def close(self):
        await self._stack.aclose()


In [99]:
host = MCPHost()

print("Connecting to math server…")
await host.add_server(sys.executable, ["math_server_http.py"])

print("Connecting to text server…")
await host.add_server(sys.executable, ["text_server_http.py"])

print(f"\nHost now knows {len(host.openai_tools)} tools from {len(set(host.session_by_tool.values()))} servers.")

Connecting to math server…
  connected — tools: ['add', 'multiply', 'power', 'modulo', 'divide']
Connecting to text server…
  connected — tools: ['uppercase', 'lowercase', 'word_count', 'reverse']

Host now knows 9 tools from 2 servers.


In [106]:
host = MCPHost_http()

print("Connecting to math server…")
await host.add_server("http://127.0.0.1:8001/mcp")

print("Connecting to text server…")
await host.add_server("http://127.0.0.1:8002/mcp")

result=await host.call("multiply",{"a":6,"b":7})

print(f"result for multilying 6 by 7 is {result}")

print(f"\nHost now knows {len(host.openai_tools)} tools from {len(set(host.session_by_tool.values()))} servers.")

Connecting to math server…
  connected — tools: ['add', 'multiply', 'power', 'modulo', 'divide']
Connecting to text server…
  connected — tools: ['uppercase', 'lowercase', 'word_count', 'reverse']
tool name which is asked for calling is multiply
result for multilying 6 by 7 is 42.0

Host now knows 9 tools from 2 servers.


In [107]:
# Let's peek at what the unified tool list looks like — this is exactly
# what we will pass to OpenAI. Nothing hand-written.
from pprint import pprint
pprint(host.openai_tools)

[{'description': 'Add two numbers.',
  'name': 'add',
  'parameters': {'properties': {'a': {'title': 'A', 'type': 'number'},
                                'b': {'title': 'B', 'type': 'number'}},
                 'required': ['a', 'b'],
                 'title': 'addArguments',
                 'type': 'object'},
  'type': 'function'},
 {'description': 'Multiply two numbers.',
  'name': 'multiply',
  'parameters': {'properties': {'a': {'title': 'A', 'type': 'number'},
                                'b': {'title': 'B', 'type': 'number'}},
                 'required': ['a', 'b'],
                 'title': 'multiplyArguments',
                 'type': 'object'},
  'type': 'function'},
 {'description': 'Raise base to the given exponent.',
  'name': 'power',
  'parameters': {'properties': {'base': {'title': 'Base', 'type': 'number'},
                                'exponent': {'title': 'Exponent',
                                             'type': 'number'}},
                 'required

In [108]:
import json
from openai import OpenAI

openai_client = OpenAI(api_key=api_key)
MODEL = "gpt-5-nano"


async def chat(host, user_message, verbose=True):
    """Run one user message through an MCP-backed tool-use loop."""
    input_items = [{"role": "user", "content": user_message}]

    while True:
        response = openai_client.responses.create(
            instructions="You must use the provided tools for all operations if possible. Do not compute anything yourself, even trivial values. Chain tools when needed.",
            model=MODEL,
            input=input_items,
            tools=host.openai_tools,
        )

        # Collect any tool calls the model emitted this turn
        tool_calls = [item for item in response.output if item.type == "function_call"]
        
        if not tool_calls:
            # No more tool calls — the model is done; return its text reply
            print("No more tool calls — the model is done; returning its text reply.")
            return response.output_text

        # For each tool call: execute via the host and append result
        for tc in tool_calls:
            args = json.loads(tc.arguments)
            if(verbose):
                print(f"🔧 {tc.name}({args})")
            result = await host.call(tc.name, args)
            if verbose:
                print(f"   → {result}")

            # Echo the function_call back, then supply its output
            input_items.append({
                "type": "function_call",
                "call_id": tc.call_id,
                "name": tc.name,
                "arguments": tc.arguments,
            })
            input_items.append({
                "type": "function_call_output",
                "call_id": tc.call_id,
                "output": result,
            })



In [71]:
answer = await chat(host, "What is 13 times 29, then raised to the power of 2 and then multipled further by 6?",True)
print(f"\n💬 {answer}")

entering now to handle tools!
🔧 multiply({'a': 13, 'b': 29})
tool name which is asked for calling is multiply
   → 377.0
entering now to handle tools!
🔧 power({'base': 377, 'exponent': 2})
tool name which is asked for calling is power
   → 142129.0
entering now to handle tools!
🔧 multiply({'a': 142129, 'b': 6})
tool name which is asked for calling is multiply
   → 852774.0
entering now to handle tools!
No more tool calls — the model is done; returning its text reply.

💬 The result is 852,774.


In [75]:
answer = await chat(host, "Reverse the string 'MODEL Context PROTCOL' and then lower case it.")
print(f"\n💬 {answer}")


🔧 reverse({'text': 'MODEL Context PROTCOL'})
tool name which is asked for calling is reverse
   → LOCTORP txetnoC LEDOM
🔧 lowercase({'text': 'LOCTORP txetnoC LEDOM'})
tool name which is asked for calling is lowercase
   → loctorp txetnoc ledom
No more tool calls — the model is done; returning its text reply.

💬 loctorp txetnoc ledom


In [109]:
answer = await chat(
    host,
    "Count the words in the sentence 'MCP makes tools discoverable at runtime' "
    "and then multiply that count by 13.5 and then raise it to the power of 3 and then divide it by 7."
)
print(f"\n💬 {answer}")


🔧 word_count({'text': 'MCP makes tools discoverable at runtime'})
tool name which is asked for calling is word_count
   → 6
🔧 multiply({'a': 6, 'b': 13.5})
tool name which is asked for calling is multiply
   → 81.0
🔧 power({'base': 81, 'exponent': 3})
tool name which is asked for calling is power
   → 531441.0
🔧 divide({'a': 531441, 'b': 7})
tool name which is asked for calling is divide
   → 75920.14285714286
No more tool calls — the model is done; returning its text reply.

💬 The result is 75920.14285714286.


In [ ]:
#pip install openai-agents

  Using cached websockets-16.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.8/846.8 kB 18.0 MB/s  0:00:00
Using cached websockets-16.0-cp313-cp313-macosx_11_0_arm64.whl (175 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [openai-agents]0m [openai-agents]
Note: you may need to restart the kernel to use updated packages.


In [104]:
# ══════════════════════════════════════════════════════════════
# OpenAI Agents SDK + MCP — the cleanest approach
# Same math_server.py + text_server.py, all three tests, zero routing code.
# ══════════════════════════════════════════════════════════════
from agents import Agent, Runner
from agents.mcp import MCPServerStdio


async def agents_sdk_demo():
    # Point the Agents SDK at BOTH of our MCP servers.
    math_mcp = MCPServerStdio(
        params={"command": sys.executable, "args": ["math_server_http.py"]},
        cache_tools_list=True,
    )
    text_mcp = MCPServerStdio(
        params={"command": sys.executable, "args": ["text_server_http.py"]},
        cache_tools_list=True,
    )

    # The SDK manages the connection lifecycle for us.
    async with math_mcp, text_mcp:
        agent = Agent(
            name="MCP Demo Assistant",
            model=MODEL,
            instructions=(
                "You have access to math and text tools over MCP. "
                "Use them for every operation — do not compute anything yourself, "
                "even trivial values. Chain tools when needed."
            ),
            mcp_servers=[math_mcp, text_mcp],
        )

        questions = [
            # Test 1 — math only
            "What is 13 times 29, then raised to the power of 2?",
            # Test 2 — text only
            "Reverse the string 'model context protocol' and then shout it.",
            # Test 3 — crosses both servers
            "Count the words in the sentence 'MCP makes tools discoverable at runtime' "
            "and then multiply that count by 13.5.",
        ]

        for i, q in enumerate(questions, 1):
            print(f"\n────────── Test {i} ──────────")
            print(f"❓ {q}")
            result = await Runner.run(starting_agent=agent, input=q)
            print(f"💬 {result.final_output}")


# Top-level await — the notebook already has a running event loop,
# so `asyncio.run(...)` would raise. This mirrors the earlier cells.
await agents_sdk_demo()


────────── Test 1 ──────────
❓ What is 13 times 29, then raised to the power of 2?
💬 377; 377^2 = 142129. Final result: 142129.

────────── Test 2 ──────────
❓ Reverse the string 'model context protocol' and then shout it.
💬 LOCOTORP TXETNOC LEDOM

────────── Test 3 ──────────
❓ Count the words in the sentence 'MCP makes tools discoverable at runtime' and then multiply that count by 13.5.
💬 81.0


In [113]:
# ══════════════════════════════════════════════════════════════
# OpenAI Agents SDK + MCP over HTTP — connect to pre-started servers                         
# Requires the HTTP bonus servers to be running (math_server_http.py on 8001,              
# text_server_http.py on 8002), either in terminals or via the Popen cell.                   
# ══════════════════════════════════════════════════════════════                             
from agents import Agent, Runner                                                             
from agents.mcp import MCPServerStreamableHttp                                               
                                                                                            
MATH_URL = "http://127.0.0.1:8001/mcp"                                                      
TEXT_URL = "http://127.0.0.1:8002/mcp"
                                                                                            
                                                                                            
async def agents_sdk_http_demo():
    # Note: `params={"url": ...}` — no command, no args. The SDK just opens                  
    # a streamable-http session to the URL. Servers are fully independent.                   
    math_mcp = MCPServerStreamableHttp(                                                      
        params={"url": MATH_URL},                                                            
        cache_tools_list=True,                                                               
    )                                                                                        
    text_mcp = MCPServerStreamableHttp(                                                    
        params={"url": TEXT_URL},
        cache_tools_list=True,                                                               
    )
                                                                                            
    # `async with` still manages the CLIENT session lifecycle,                               
    # but the server processes keep running after this block exits.
    async with math_mcp, text_mcp:                                                           
        agent = Agent(                                                                     
            name="MCP Demo Assistant",                                                       
            model=MODEL,                                                                   
            instructions=(                                                                   
                "You have access to math and text tools over MCP. "
                "Use them for every operation — do not compute anything yourself, "          
                "even trivial values. Chain tools when needed."                            
            ),                                                                               
            mcp_servers=[math_mcp, text_mcp],
        )                                                                                    
                                                                                            
        questions = [
            "What is 13 times 29, then raised to the power of 2?",
            "Reverse the string 'model context protocol' and then shout it.",                
            "Count the words in the sentence 'MCP makes tools discoverable at runtime' "
            "and then multiply that count by 13.5.",                                         
        ]                                                                                    

        for i, q in enumerate(questions, 1):                                                 
            print(f"\n────────── Test {i} ──────────")                                     
            print(f"❓ {q}")
            result = await Runner.run(starting_agent=agent, input=q)                         
            print(f"💬 {result.final_output}")
                                                                                            
                                                                                            
await agents_sdk_http_demo()


────────── Test 1 ──────────
❓ What is 13 times 29, then raised to the power of 2?
💬 142129

────────── Test 2 ──────────
❓ Reverse the string 'model context protocol' and then shout it.
💬 LOCOTORP TXETNOC LEDOM

────────── Test 3 ──────────
❓ Count the words in the sentence 'MCP makes tools discoverable at runtime' and then multiply that count by 13.5.
💬 81.0


In [ ]:
host = MCPHost_http()
try:
    
    print("Connecting to math server…")
    await host.add_server("http://127.0.0.1:8001/mcp")

    print("Connecting to text server…")
    await host.add_server("http://127.0.0.1:8002/mcp")

    result=await host.call("multiply",{"a":6,"b":7})

    print(f"result for multilying 6 by 7 is {result}")

    print(f"\nHost now knows {len(host.openai_tools)} tools from {len(set(host.session_by_tool.values()))} servers.")
finally:
    await host.close()

Connecting to math server…
  connected — tools: ['add', 'multiply', 'power', 'modulo', 'divide']
Connecting to text server…
  connected — tools: ['uppercase', 'lowercase', 'word_count', 'reverse']
tool name which is asked for calling is multiply
result for multilying 6 by 7 is 42.0

Host now knows 9 tools from 2 servers.


In [ ]:
# Let's peek at what the unified tool list looks like — this is exactly
# what we will pass to OpenAI. Nothing hand-written.
from pprint import pprint
pprint(host.openai_tools)

[{'description': 'Add two numbers.',
  'name': 'add',
  'parameters': {'properties': {'a': {'title': 'A', 'type': 'number'},
                                'b': {'title': 'B', 'type': 'number'}},
                 'required': ['a', 'b'],
                 'title': 'addArguments',
                 'type': 'object'},
  'type': 'function'},
 {'description': 'Multiply two numbers.',
  'name': 'multiply',
  'parameters': {'properties': {'a': {'title': 'A', 'type': 'number'},
                                'b': {'title': 'B', 'type': 'number'}},
                 'required': ['a', 'b'],
                 'title': 'multiplyArguments',
                 'type': 'object'},
  'type': 'function'},
 {'description': 'Raise base to the given exponent.',
  'name': 'power',
  'parameters': {'properties': {'base': {'title': 'Base', 'type': 'number'},
                                'exponent': {'title': 'Exponent',
                                             'type': 'number'}},
                 'required

In [ ]:
import json
from openai import OpenAI

openai_client = OpenAI(api_key=api_key)
MODEL = "gpt-5-nano"


async def chat(host, user_message, verbose=True):
    """Run one user message through an MCP-backed tool-use loop."""
    input_items = [{"role": "user", "content": user_message}]

    while True:
        response = openai_client.responses.create(
            instructions="You must use the provided tools for all operations if possible. Do not compute anything yourself, even trivial values. Chain tools when needed.",
            model=MODEL,
            input=input_items,
            tools=host.openai_tools,
        )

        # Collect any tool calls the model emitted this turn
        tool_calls = [item for item in response.output if item.type == "function_call"]
        
        if not tool_calls:
            # No more tool calls — the model is done; return its text reply
            print("No more tool calls — the model is done; returning its text reply.")
            return response.output_text

        # For each tool call: execute via the host and append result
        for tc in tool_calls:
            args = json.loads(tc.arguments)
            if(verbose):
                print(f"🔧 {tc.name}({args})")
            result = await host.call(tc.name, args)
            if verbose:
                print(f"   → {result}")

            # Echo the function_call back, then supply its output
            input_items.append({
                "type": "function_call",
                "call_id": tc.call_id,
                "name": tc.name,
                "arguments": tc.arguments,
            })
            input_items.append({
                "type": "function_call_output",
                "call_id": tc.call_id,
                "output": result,
            })



---
## Bonus — Consuming a third-party MCP server (`mcp-server-git`)

So far every server was **ours**. The other half of MCP's promise is plugging in **servers someone else wrote** — no integration code, no custom schemas, no API-key dance. Our existing client just spawns their binary and talks protocol.

We'll use the official Git server: `mcp-server-git`. It exposes tools like `git_status`, `git_log`, `git_diff`, `git_show` — all scoped to a repo path we give it. No auth needed: it just shells out to `git` locally.

```bash
pip install mcp-server-git   # one-time
```

**One caveat about "starting the server yourself":** `mcp-server-git` is **stdio-only** — it reads JSON-RPC from stdin and writes to stdout. That means a process running in a terminal can't be talked to by another process; stdio requires the parent to spawn the child. So there are two realistic options:

| Option | You do | Notebook does |
|---|---|---|
| **A. You own the repo, notebook spawns the server** | clone / pick a real repo | spawn `python -m mcp_server_git` as a stdio subprocess pointing at your repo |
| **B. You own the repo AND the server process** (via `mcp-proxy`) | `mcp-proxy --sse-port 8765 -- python -m mcp_server_git --repository <path>` | connect via SSE to `http://127.0.0.1:8765/sse` |

Option A is what we'll do by default. Option B is at the bottom if you truly want the server as a standalone process.

In [ ]:
#pip install mcp-server-git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [mcp-server-git]
Note: you may need to restart the kernel to use updated packages.


In [115]:
# --- Step 1: clone a real online repo (or point at one you already have) --
# Pick any small public repo. octocat/Hello-World is tiny and canonical.
# If you've already cloned something elsewhere, just set REPO_PATH to it.

import subprocess, pathlib

REPO_URL  = "https://github.com/octocat/Hello-World.git"
REPO_PATH = pathlib.Path("/Users/anupagarwal/Documents/PersonalFolder/ScalerAICourse/GitMCPTesting/TestGitCLone")

if not (REPO_PATH / ".git").exists():
    print(f"Cloning {REPO_URL} → {REPO_PATH} …")
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--quiet", REPO_URL, str(REPO_PATH)],
        check=True,
    )
else:
    print(f"Using existing clone at {REPO_PATH}")

# Confirm we're looking at a real repo with real commits.
log_preview = subprocess.run(
    ["git", "log", "--oneline", "-n", "3"],
    cwd=REPO_PATH, check=True, capture_output=True, text=True,
).stdout
print(f"\nMost recent commits:\n{log_preview}")

Cloning https://github.com/octocat/Hello-World.git → /Users/anupagarwal/Documents/PersonalFolder/ScalerAICourse/GitMCPTesting/TestGitCLone …

Most recent commits:
7fd1a60 Merge pull request #6 from Spaceghost/patch-1
7629413 New line at end of file. --Signed off by Spaceghost
553c207 first commit



In [116]:
# --- Step 2: spawn mcp-server-git and talk protocol -----------------------
# Same client pattern as Step 3 of this notebook. Only the
# StdioServerParameters change — our code has no idea it's "git".
#
# Note: every mcp-server-git tool takes `repo_path` as a required argument
# in addition to the `--repository` CLI flag. Quirk of that server.

git_params = StdioServerParameters(
    command=sys.executable,
    args=["-m", "mcp_server_git", "--repository", str(REPO_PATH)],
)

async with stdio_client(git_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        # 1) DISCOVERY — what does this server expose?
        listed = await session.list_tools()
        print(f"mcp-server-git exposes {len(listed.tools)} tools:\n")
        for t in listed.tools:
            first_line = (t.description or "").split("\n")[0][:70]
            print(f"  • {t.name:22} — {first_line}")

        def show(label, result, limit=600):
            text = "\n".join(c.text for c in result.content if hasattr(c, "text"))
            print(f"\n── {label} ──")
            print(text[:limit] + ("…" if len(text) > limit else ""))

        # 2) git_status — working tree state
        show("git_status",
             await session.call_tool("git_status", {"repo_path": str(REPO_PATH)}))

        # 3) git_log — last 5 commits from a real public repo
        show("git_log (last 5)",
             await session.call_tool("git_log",
                                     {"repo_path": str(REPO_PATH), "max_count": 5}))

        # 4) git_show — full details of HEAD (commit message + diff)
        show("git_show HEAD",
             await session.call_tool("git_show",
                                     {"repo_path": str(REPO_PATH), "revision": "HEAD"}))

mcp-server-git exposes 12 tools:

  • git_status             — Shows the working tree status
  • git_diff_unstaged      — Shows changes in the working directory that are not yet staged
  • git_diff_staged        — Shows changes that are staged for commit
  • git_diff               — Shows differences between branches or commits
  • git_commit             — Records changes to the repository
  • git_add                — Adds file contents to the staging area
  • git_reset              — Unstages all staged changes
  • git_log                — Shows the commit logs
  • git_create_branch      — Creates a new branch from an optional base branch
  • git_checkout           — Switches branches
  • git_show               — Shows the contents of a commit
  • git_branch             — List Git branches

── git_status ──
Repository status:
On branch master
Your branch is up to date with 'origin/master'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.DS_Store

n

---
### Option B (optional) — You start the server yourself in a terminal

If you really want `mcp-server-git` to live as an independent process, you need to put HTTP on the outside of it. The common way is **`mcp-proxy`** — a tiny utility that wraps any stdio MCP server and re-exposes it over SSE. The stdio server doesn't know anything changed; your client just talks HTTP/SSE to the proxy.

**Install once:**
```bash
pip install mcp-proxy
```

**Start it yourself in a terminal** (leave it running):
```bash
mcp-proxy --sse-port 8765 \
  -- python -m mcp_server_git --repository /tmp/mcp_external_repo
# -> INFO:  Uvicorn running on http://127.0.0.1:8765
```

Everything after `--` is the stdio command that `mcp-proxy` will spawn internally and relay for. You can `Ctrl-C` it, change repos, restart — the notebook reconnects on the next call.

**Then connect from the notebook via SSE** (not `streamablehttp_client` — `mcp-proxy` speaks the older SSE transport by default):

```python
from mcp.client.sse import sse_client

SSE_URL = "http://127.0.0.1:8765/sse"

async with sse_client(SSE_URL) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        listed = await session.list_tools()
        print("Tools via mcp-proxy:", [t.name for t in listed.tools])
        result = await session.call_tool(
            "git_log",
            {"repo_path": str(REPO_PATH), "max_count": 3},
        )
        print(result.content[0].text[:400])
```

Conceptually you now have the same architecture as the HTTP transport bonus above, but for a third-party stdio-only server. That's the pattern you'll use in real deployments whenever you want to host a stdio server behind a URL.

> VPN / proxy env note: if `streamable_http` hung for you earlier, SSE will hang for the same reason. Keep the `NO_PROXY` / proxy-unset snippet from that cell in the kernel.

In [ ]:
#pip install mcp-proxy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [mcp-proxy]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# !mcp-proxy --sse-port 8765 -- python -m mcp_server_git --repository /Users/anupagarwal/Documents/PersonalFolder/ScalerAICourse/GitMCPTesting/TestGitCLone

[I 2026-06-16 19:52:50,023.023 mcp_proxy.__main__] Configured default server: python -m mcp_server_git --repository /Users/anupagarwal/Documents/PersonalFolder/ScalerAICourse/GitMCPTesting/TestGitCLone
[I 2026-06-16 19:52:50,024.024 mcp_proxy.mcp_server] Setting up default server: python -m mcp_server_git --repository /Users/anupagarwal/Documents/PersonalFolder/ScalerAICourse/GitMCPTesting/TestGitCLone
[I 2026-06-16 19:52:50,829.829 mcp.server.streamable_http_manager] StreamableHTTP session manager started
[I 2026-06-16 19:52:50,830.830 mcp_proxy.mcp_server] Serving MCP Servers via SSE:
[I 2026-06-16 19:52:50,830.830 mcp_proxy.mcp_server]   - http://127.0.0.1:0/sse
INFO:     Started server process [18474]
INFO:     Waiting for application startup.
[I 2026-06-16 19:52:50,848.848 mcp_proxy.mcp_server] Main application lifespan starting...
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:64780 (Press CTRL+C to quit)
^C
INFO:     Finished server process

In [127]:
from mcp.client.sse import sse_client

SSE_URL = "http://127.0.0.1:64892/sse"

async with sse_client(SSE_URL) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        listed = await session.list_tools()
        print("Tools via mcp-proxy:", [t.name for t in listed.tools])
        result = await session.call_tool(
            "git_log",
            {"repo_path": str(REPO_PATH), "max_count": 3},
        )
        print(result.content[0].text[:400])

Tools via mcp-proxy: ['git_status', 'git_diff_unstaged', 'git_diff_staged', 'git_diff', 'git_commit', 'git_add', 'git_reset', 'git_log', 'git_create_branch', 'git_checkout', 'git_show', 'git_branch']
Commit history:
Commit: '7fd1a60b01f91b314f59955a4e4d4e80d8edf11d'
Author: <git.Actor "The Octocat <octocat@nowhere.com>">
Date: 2012-03-06 15:06:50-08:00
Message: 'Merge pull request #6 from Spaceghost/patch-1\n\nNew line at end of file.'

Commit: '762941318ee16e59dabbacb1b4049eec22f0d303'
Author: <git.Actor "Johnneylee Jack Rollins <Johnneylee.rollins@gmail.com>">
Date: 2011-09-13 21:42:41-07:00
